# v3.6 T2-Weighted TSE Template

This notebook shows software-defined contrast scheduling on top of a GUI-loaded TSE sequence. The GUI owns sequence selection and visualization; each contrast point receives a transient sequence edit and its own Python-governed TR.

In [ ]:
import importlib
from IPython.display import display

import src_v3_6.app as gui
from src_v3_6 import core_api
gui = importlib.reload(gui)

display(gui.GUI_total)

## Select And Inspect The Virgin Sequence

In the GUI, open **Sequence -> Sequence Files**, load the TSE sequence you want to visualize and run, then inspect the timeline/pulse view. The notebook reads that currently loaded sequence only for sanity checking.

In [ ]:
virgin_seq = gui.get_current_sequence()

effective_te_delays_us = [8000, 12000, 16000, 20000]
tr_schedule_ms = [1000, 1500, 2000, 10000]
averages_per_contrast = 1

virgin_seq['ExpConfig']

In [ ]:
# Initialise FPGA from the GUI before this cell.
results_by_te = {}

def make_te_transform(delay_us):
    def transform(seq, context):
        core_api.set_section_delay(seq, 'Delay_TE_2', delay_us)
        return seq
    return transform

for point_idx, delay_us in enumerate(effective_te_delays_us):
    config = gui.make_experiment_config(
        num_averages=averages_per_contrast,
        loop_count=1,
        tr_ms=tr_schedule_ms[point_idx],
        python_tr_enabled=True,
    )

    def on_scan(result, delay_us=delay_us):
        print(f'TE delay {delay_us} us: scan {result.context.scan_idx + 1}/{result.context.total_scans}')

    avg_ch0, avg_ch1 = gui.run_api_sequence(
        config=config,
        sequence_transform=make_te_transform(delay_us),
        scan_callback=on_scan,
    )
    results_by_te[delay_us] = (avg_ch0, avg_ch1)

list(results_by_te.keys())